# Допуск: Visualizing and Understanding Recurrent Networks

**Статья:** A. Karpathy, J. Johnson, L. Fei-Fei, 2015 — <https://arxiv.org/abs/1506.02078>

5 заданий. Под заголовком каждого задания указан его тип:

- **что есть что** — названия функций или классов заменены: определите, что чему
  соответствует, и ответьте на вопросы;
- **расхождение со статьёй** — код реализует не то, что описано в статье: найдите это место;
- **перепутаны местами блоки кода** — какой-то блок стоит не на своём месте: найдите его;
- **неверный комментарий** — код корректен, но один комментарий утверждает неправду: найдите
  его.

Для найденного отклонения напишите: где оно, что не так, какое место статьи (раздел /
формула / таблица / цитата) нарушено и как должно быть. Код писать не нужно — достаточно
объяснить словами. Запускать и отлаживать код тоже не нужно: все задания решаются чтением
кода и статьи.

Общие импорты:

In [ ]:
import bisect, copy, string
from collections import Counter, defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F

Сниппеты — части одной реализации: `tbptt_chunks(streams)` нарезает корпус на куски по 100 символов, `LSTMCell` — LSTM-ячейка из §3.1. Не показанные функции считайте корректными.

## Задание 1. Три рекуррентные ячейки

**Тип задания:** что есть что — названия функций/классов заменены, определите, что чему соответствует.

Три класса — vanilla RNN, LSTM и GRU из §3.1. `x` — вход снизу (h^{l−1}_t), `h` / `state` —
состояние этого слоя на предыдущем шаге. Код корректен.

**Определите, какой класс какой ячейке соответствует, и ответьте на вопросы.**

In [ ]:
class CellA(nn.Module):
    def __init__(self, n):
        super().__init__()
        self.Wr = nn.Linear(2 * n, 2 * n)
        self.Wx = nn.Linear(n, n)
        self.Wg = nn.Linear(n, n)
    def forward(self, x, h):
        r, z = self.Wr(torch.cat([x, h], -1)).sigmoid().chunk(2, -1)
        h_cand = torch.tanh(self.Wx(x) + self.Wg(r * h))
        return (1 - z) * h + z * h_cand

class CellB(nn.Module):
    def __init__(self, n):
        super().__init__()
        self.W = nn.Linear(2 * n, n)
    def forward(self, x, h):
        return torch.tanh(self.W(torch.cat([x, h], -1)))

class CellC(nn.Module):
    def __init__(self, n):
        super().__init__()
        self.W = nn.Linear(2 * n, 4 * n)
    def forward(self, x, state):
        h, c = state
        i, f, o, g = self.W(torch.cat([x, h], -1)).chunk(4, -1)
        i, f, o, g = i.sigmoid(), f.sigmoid(), o.sigmoid(), g.tanh()
        c = f * c + i * g
        return o * c.tanh(), c

**Вопросы.**

1. Через какую строку градиент может проходить назад во времени «почти без изменений»? Почему
   авторы называют аддитивное обновление *«a critical feature of the LSTM's design»*?
2. Почему у `g` активация `tanh`, а у гейтов — `sigmoid`?
3. Про GRU первого слоя авторы пишут, что он работает в *«purely feed-forward mode»* (r редко
   близок к 1, z редко близок к 0). Покажите по коду, что происходит при r → 0, z → 1.
4. В статье h^0_t = x_t (one-hot размерности K). Сколько параметров у матрицы W первого
   LSTM-слоя при K = 87, n = 512? Сравните с «≈ 1.3M» для однослойной LSTM на 512 ячеек (§4.1).

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 2. Одна эпоха обучения

**Тип задания:** расхождение со статьёй — код реализует не то, что описано в статье.

Модель получает очередной кусок и скрытое состояние (список пар `(h, c)` по слоям).

**В коде одно отклонение от статьи — найдите его.**

In [ ]:
def train_epoch(model, streams, opt):
    state = None
    for x, y in tbptt_chunks(streams):
        state = None                                     # every chunk starts from a clean state
        logits, state = model(x, state)                  # (B, T, K)
        loss = F.cross_entropy(logits.flatten(0, 1), y.flatten())
        opt.zero_grad()
        loss.backward()
        opt.step()

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 3. Глубокая сеть, развёрнутая во времени

**Тип задания:** расхождение со статьёй — код реализует не то, что описано в статье.

Сетка состояний h^l_t из §3.1 (время × глубина) на LSTM-ячейках; на выходе — логиты
следующего символа (§3.2).

**В коде одно отклонение от статьи — найдите его.**

In [ ]:
class CharLSTM(nn.Module):
    def __init__(self, K, n, L=2):
        super().__init__()
        self.inp = nn.Embedding(K, n)                    # 1-of-K input times a matrix
        self.cells = nn.ModuleList(LSTMCell(n) for _ in range(L))
        self.Wy = nn.Linear(n, K)                        # y_t = W_y h^L_t
    def forward(self, x, state=None):                    # x: (B, T) char ids
        B, T = x.shape
        if state is None:
            z = torch.zeros(B, self.Wy.in_features, device=x.device)
            state = [(z, z)] * len(self.cells)
        state, ys = list(state), []
        for t in range(T):
            h = self.inp(x[:, t])                        # h^0_t
            for l, cell in enumerate(self.cells):
                h, c = cell(h, state[l])                 # h^{l-1}_t, (h^l_{t-1}, c^l_{t-1})
                state[l] = (h, c)
            ys.append(self.Wy(c))
        return torch.stack(ys, 1), state                 # (B, T, K) logits

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 4. Визуализация ячеек и детектор кавычек

**Тип задания:** неверный комментарий — код корректен, но один комментарий утверждает неправду.

Снимаем активации ячеек при проходе по тексту (как на Fig. 2) и ищем ячейку, которая сильнее
всего связана с признаком «внутри кавычек».

**В коде одно отклонение — найдите его.**

In [ ]:
@torch.no_grad()
def cell_trace(model, ids, layer=-1):
    # feed chars one by one; record h = o * tanh(c), the quantity colored in Fig. 2
    state, trace = None, []
    for t in range(len(ids)):
        _, state = model(ids[t].view(1, 1), state)
        h, c = state[layer]
        trace.append(c.tanh()[0])                        # (n,)
    return torch.stack(trace)                            # (T, n)

def inside_quotes(text):
    flags, inside = [], False
    for ch in text:
        inside = not inside if ch == '"' else inside
        flags.append(float(inside))
    return torch.tensor(flags)                           # (T,)

def find_cell(trace, feature):
    # cell whose activation correlates best with the feature
    a = (trace - trace.mean(0)) / (trace.std(0) + 1e-8)
    b = (feature - feature.mean()) / (feature.std() + 1e-8)
    corr = (a * b[:, None]).mean(0)                      # (n,)
    return corr.abs().argmax().item(), corr

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 5. Анализ ошибок: «peel the onion»

**Тип задания:** перепутаны местами блоки кода.

Ошибки LSTM снимаются оракулами по очереди (§4.4, Fig. 6). Маски `memory_mask` и `rare_mask`
посчитаны заранее; `p_ngram` — вероятности правильного символа у 1..9-граммных моделей.

**В коде одно отклонение от статьи — найдите его.**

In [ ]:
def peel_onion(err, text, p_ngram, p_lstm, memory_mask, rare_mask):
    # err: (T,) bool LSTM errors; p_ngram: (9, T), P(correct) of 1..9-gram models
    after_space = torch.tensor([t > 0 and text[t - 1] in ' "\n' for t in range(len(text))])
    oracles = [
        ("boost", p_lstm + 0.1 > 0.5),
        ("n-gram", (p_ngram > 0.5).any(0)),
        ("dynamic memory", memory_mask),
        ("rare words", rare_mask),
        ("word model", after_space),
        ("punctuation", torch.tensor([ch in string.punctuation for ch in text])),
    ]
    removed = {}
    for name, mask in oracles:
        removed[name] = int((err & mask).sum())
        err = err & ~mask
    return removed, int(err.sum())

**Ваш ответ:**

_Напишите ответ здесь._